In [ ]:
"""
This module implements training and evaluation of a multi-layer perceptron in PyTorch.
You should fill in code into indicated sections.
"""
from __future__ import absolute_import
from __future__ import division
from __future__ import print_function

import os
import numpy as np
import json
import math

import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
sns.set()

from copy import deepcopy

# Progress bar
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data

import tensorboard as tb
from torch.utils.tensorboard import SummaryWriter

In [ ]:
%load_ext tensorboard

In [ ]:
class MLP(nn.Module):

    def __init__(self, n_inputs, n_hidden, n_classes, use_batch_norm=False):
        """ Initializing the MLP module """
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(),
            nn.ReLu(),
            nn.MaxPool2d(2),
            nn.Conv2d(),
            nn.ReLu(),
            nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(),
            nn.ReLU(),
            nn.Linear(128, 80)
        )

    def forward(self, x):
        out = self.net(x)
        return out

    @property
    def device(self):
        """
        Returns the device on which the model is. Can be useful in some situations.
        """
        return next(self.parameters()).device

In [ ]:
def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available(): # GPU operation have separate seed
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

# Additionally, some operations on a GPU are implemented stochastic for efficiency
# We want to ensure that all operations are deterministic on GPU (if used) for reproducibility
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Fetching the device that will be used throughout this notebook
device = torch.device("cpu") if not torch.cuda.is_available() else torch.device("cuda:0")
print("Using device", device)

In [ ]:
def accuracy(predictions, targets):

    preds = predictions.argmax(dim=1)
    correct = (preds == targets)
    accuracy = correct.float().mean().item()
    return accuracy

def eval_model(model, data_loader):

    model.eval()

    total_correct = 0
    total_samples = 0

    with torch.no_grad():
        for inputs, targets in data_loader:
            outputs = model(inputs)

            preds = outputs.argmax(dim=1)

            total_correct += (preds == targets).sum().item()
            total_samples += targets.size(0)

    avg_accuracy = total_correct / total_samples

    return avg_accuracy